# stockout — the five questions

The questions were committed in [`docs/questions.md`](../docs/questions.md) **before** this
notebook existed, so the findings cannot be retrofitted to whatever the charts happened to
show. Each one names what would count as a *no*. See
[ADR 0006](../docs/decisions/0006-analysis-in-notebooks.md) for why the analysis lives
here and the plumbing does not.

> **Every number below came from `stockout.data.synth`.** The real Rossmann archive needs
> a Kaggle account and an accepted competition-rules page. This repository's rule is that
> the generator *tests machinery and never supports a finding*: what follows is evidence
> that the pipeline runs and produces coherent output, and is not evidence about retail.
> Run `python -m stockout fetch`, point `SALES` and `STORES` at the downloaded files, and
> every cell answers the same question against data that can carry an answer.

Answers are written up in [`docs/results.md`](../docs/results.md), with a verdict each —
held, lost, or inconclusive.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from stockout import config, plots
from stockout.data import schemas as s
from stockout.dataset import prepare
from stockout.evaluate import metrics
from stockout.evaluate.backtest import backtest, summarise
from stockout.evaluate.classification import confusion
from stockout.evaluate.classification import score as score_classes
from stockout.evaluate.comparison import compare
from stockout.models.baselines import SeasonalNaive
from stockout.models.registry import build
from stockout.split.strategies import time_holdout
from stockout.targets import DEMAND_CLASS_CODE, add_demand_class, class_balance, fit_thresholds

plots.apply_style()

#: Point these at the downloaded Rossmann files to answer the questions for real.
SALES = config.SAMPLE_PATH
STORES = config.SAMPLE_STORES_PATH

#: The supplier's reorder cycle. Q1 varies it; everything else holds it here.
HORIZON = 7

#: One held-out window, used by Q3 and Q5, with a horizon-sized gap in front of it so it
#: is scoring a seven-day-ahead forecast rather than a one-day-ahead one.
TEST_DAYS = 28
GAP_DAYS = HORIZON

prepared = prepare(sales_path=SALES, stores_path=STORES, horizon=HORIZON)
frame = prepared.frame
print(prepared.summary())

## What is in the frame

`dataset.prepare` reads both files, joins them, derives the date-relative store features,
builds the calendar and lag columns, sorts date-major and drops the warm-up rows no
rolling window can fill. Six steps, one order, one function — see its docstring for what
goes wrong when a caller reimplements them.

In [ ]:
train, test = time_holdout(frame, test_days=TEST_DAYS, gap_days=GAP_DAYS)

print(f"stores          {frame[s.STORE].nunique()}")
print(f"calendar        {frame[s.DATE].min().date()} to {frame[s.DATE].max().date()}")
print(f"train / test    {len(train):,} / {len(test):,} rows")
print(f"trading rows    {int((frame[s.OPEN] == 1).sum()):,}")

balance = pd.DataFrame(
    {
        "training window": class_balance(add_demand_class(train, fit_thresholds(train))),
        "test window": class_balance(add_demand_class(test, fit_thresholds(train))),
    }
)
print("\ndemand class balance (cut points fitted on the training window only)")
print(balance.round(3).to_string())

The training balance is thirds **by construction** — the cut points are that window's own
terciles. The test window has drifted, and that drift is the reason macro-F1 leads every
classification table here and accuracy follows it: a model can drop the smallest class
entirely, score respectably on accuracy, and be useless at the job it was built for.

## The ladder — does the extra complexity pay?

Registry order, simplest first. A table sorted by score answers *which won*; this order
also answers *did the extra complexity pay*, which is the harder question to fake.

This is the table [`docs/results.md`](../docs/results.md) quotes, and
`python -m stockout compare` prints the same one.

In [ ]:
ladder = compare(frame, task="regression", test_days=TEST_DAYS, gap_days=GAP_DAYS)
print(ladder[["model", "train_rows", "r2", "wmape", "seconds"]].round(3).to_string(index=False))

best = ladder.loc[ladder["wmape"].idxmin()]
floor = ladder.loc[ladder["model"] == "linear"].iloc[0]
print(
    f"\nbest: {best['model']} at WMAPE {best['wmape']:.4f}; "
    f"a straight line scores {floor['wmape']:.4f} — a gap of {floor['wmape'] - best['wmape']:.4f}"
)

## Q1 — Does forecast accuracy decay with horizon, and how fast?

**Expected:** monotonic decay, steepest between 7 and 14 days.
**Counts as a no:** WMAPE flat across horizons, which would mean the model is predicting a
store-level average and ignoring recent history entirely.

The horizon is a business input — it is set by supplier lead time, not chosen by the
modeller — so this curve says what accuracy a given lead time buys. Each horizon needs its
own prepared frame, because the lag a model may read is defined by the horizon it forecasts
at: `features/lags.py` refuses a lag shorter than the horizon, which is the guard that
stops a "42-day forecast" quietly reading yesterday.

In [ ]:
HORIZONS = (7, 14, 28, 42)
MODEL = "ridge"

rows = []
for horizon in HORIZONS:
    at_horizon = prepare(sales_path=SALES, stores_path=STORES, horizon=horizon).frame
    for name, factory in (
        (MODEL, lambda h=horizon: build(MODEL, task="regression", horizon=h)),
        ("seasonal_naive", SeasonalNaive),
    ):
        scored = summarise(
            backtest(at_horizon, factory, n_folds=5, horizon=horizon, gap=0)
        )
        rows.append({"horizon": horizon, "model": name, "wmape": scored["wmape"]})

decay = pd.DataFrame(rows).pivot(index="horizon", columns="model", values="wmape")
print(decay.round(4).to_string())

In [ ]:
fig, ax = plt.subplots()
for name in decay.columns:
    ax.plot(decay.index, decay[name], marker="o", label=name)
ax.set_xlabel("horizon (days)")
ax.set_ylabel("mean WMAPE across 5 rolling-origin folds")
ax.set_title("Q1 — accuracy against the lead time it has to cover")
ax.set_xticks(list(HORIZONS))
ax.legend()
print(plots.save(fig, "q1_horizon_decay"))
plt.show()

## Q2 — Does the baseline beat a fitted model on low-volume stores?

**Expected:** the fitted model wins overall but loses on the quietest stores, where each
one has too few high-signal observations.
**Counts as a no:** the fitted model wins uniformly across every store.

If it holds, the right answer is a per-store model choice rather than one model — a
finding nobody puts in a tutorial. MASE is the metric because it is scaled by
seasonal-naive on the same window, so *below 1 means beaten* and the comparison needs no
further arithmetic.

Two fitted models are compared against the baseline, not one, because "the fitted model"
is ambiguous and the two answers differ:

- **pooled** — one model trained on every store, scored store by store. This is what you
  would deploy, and a quiet store benefits from the 1114 others' seasonality.
- **per store** — a separate model trained on that store's rows alone. More specialised
  and far less data, which is exactly the trade the question is about.

The committed sample has four stores, so "volume quartile" and "store" are the same thing
here. On Rossmann's 1115 the quartiles are populated and this cell groups them.

In [ ]:
VOLUME_QUARTILES = 4

trading = frame[frame[s.OPEN] == 1]
volume = trading.groupby(s.STORE, observed=True)[s.SALES].mean().sort_values()
quartile = pd.qcut(volume, min(VOLUME_QUARTILES, volume.nunique()), labels=False, duplicates="drop")

scored_rows = test[test[s.OPEN] == 1]
pooled = build(MODEL, task="regression", horizon=HORIZON).fit(train).predict(test)
naive = SeasonalNaive().fit(train).predict(test)

rows = []
for store, store_test in scored_rows.groupby(s.STORE, observed=True):
    store_train = train[train[s.STORE] == store]
    alone = build(MODEL, task="regression", horizon=HORIZON).fit(store_train).predict(store_test)
    actual = store_test[s.SALES]
    rows.append(
        {
            "store": int(str(store)),
            "mean sales": float(volume.loc[store]),
            "quartile": int(quartile.loc[store]),
            "train rows": int((store_train[s.OPEN] == 1).sum()),
            "pooled": metrics.mase(
                actual, pooled.loc[store_test.index], y_baseline=naive.loc[store_test.index]
            ),
            "per store": metrics.mase(
                actual, alone, y_baseline=naive.loc[store_test.index]
            ),
        }
    )

per_store = pd.DataFrame(rows).sort_values("mean sales").reset_index(drop=True)
print(per_store.round(3).to_string(index=False))
for column in ("pooled", "per store"):
    lost = int((per_store[column] >= 1.0).sum())
    print(f"\n{column}: seasonal-naive wins on {lost} of {len(per_store)} stores; "
          f"worst MASE {per_store[column].max():.3f}")

In [ ]:
fig, ax = plt.subplots()
positions = np.arange(len(per_store))
width = 0.38
for offset, (column, colour) in enumerate((("pooled", "#1f6f54"), ("per store", "#2c6fa8"))):
    ax.bar(positions + (offset - 0.5) * width, per_store[column], width, label=column, color=colour)
ax.axhline(1.0, color="black", linewidth=1)
ax.annotate(
    "seasonal-naive",
    xy=(0.01, 1.0),
    xycoords=("axes fraction", "data"),
    va="bottom",
    fontsize=9,
)
ax.set_xticks(positions, [str(store) for store in per_store["store"]])
ax.set_xlabel("store, quietest first")
ax.set_ylabel(f"MASE of {MODEL} (below 1 beats the baseline)")
ax.set_title("Q2 — does the fitted model win everywhere, or only on average?")
ax.legend()
print(plots.save(fig, "q2_store_volume"))
plt.show()

## Q3 — How much of the total error comes from a small number of days?

**Expected:** heavily concentrated — a small share of days carries most of the error,
clustered on holidays and promotion boundaries.
**Counts as a no:** error spread evenly across days, so there is no special case worth
building.

This decides whether effort belongs in better features or in a separate event model. The
diagonal is the null: if the worst 10% of days carried exactly 10% of the error, the curve
would lie on it.

In [ ]:
model = build(MODEL, task="regression", horizon=HORIZON).fit(train)
scored_rows = test[test[s.OPEN] == 1]
predicted = model.predict(test).loc[scored_rows.index]

errors = (scored_rows[s.SALES] - predicted).abs().sort_values(ascending=False)
share_of_error = errors.cumsum() / errors.sum()
share_of_days = np.arange(1, len(errors) + 1) / len(errors)

for fraction in (0.05, 0.10, 0.20, 0.50):
    cut = max(round(fraction * len(errors)), 1)
    print(f"worst {fraction:>4.0%} of days carry {share_of_error.iloc[cut - 1]:.1%} of the error")

columns = [s.DATE, s.STORE, s.SALES, s.PROMO, s.SCHOOL_HOLIDAY]
worst = scored_rows.loc[errors.index[:5], columns].assign(
    predicted=predicted.loc[errors.index[:5]].round(0),
    error=errors.iloc[:5].round(0),
)
print("\nthe five worst days")
print(worst.to_string(index=False))

In [ ]:
fig, ax = plt.subplots()
ax.plot(share_of_days, share_of_error.to_numpy(), linewidth=2, label="observed")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1, label="evenly spread")
ax.set_xlabel("share of test days, worst first")
ax.set_ylabel("cumulative share of total absolute error")
ax.set_title("Q3 — is the error concentrated, or is every day equally hard?")
ax.legend()
print(plots.save(fig, "q3_error_concentration"))
plt.show()

## Q4 — Does the promotion lift persist after the promotion ends, or reverse?

**Expected:** a dip. Promotions pull demand forward rather than creating it.
**Counts as a no:** post-promotion sales at or above baseline, meaning promotions
genuinely grow demand.

This one asks nothing of any model — it is a question about the data. The baseline is
matched on **store and weekday**, because Saturday outsells Tuesday by more than any
promotion does, and an unmatched comparison would measure the calendar instead.

In [ ]:
WAKE_DAYS = 7


def days_after_promo(on_promotion: np.ndarray, *, wake_days: int) -> np.ndarray:
    """Trading days since this store's last promotion day, or -1 if the row is not in a wake.

    Three rows get -1 and each for its own reason: a day *on* promotion (the counter
    resets), a day more than `wake_days` after one (out of the window), and any day
    before this store's first promotion (there is no promotion for it to be after).

    The last two are the ones an off-by-one hides. Counting through a new promotion is
    how the wake picks up the *next* cycle's lift and reports it as persistence.
    """
    since = np.full(len(on_promotion), -1)
    counter = 0
    seen_one = False
    for position, is_on in enumerate(on_promotion):
        if is_on:
            counter, seen_one = 0, True
            continue
        counter += 1
        if seen_one and counter <= wake_days:
            since[position] = counter
    return since


trading = frame[frame[s.OPEN] == 1].sort_values([s.STORE, s.DATE])
wake_index = [
    pd.Series(
        days_after_promo(store_rows[s.PROMO].to_numpy() == 1, wake_days=WAKE_DAYS),
        index=store_rows.index,
    )
    for _, store_rows in trading.groupby(s.STORE, observed=True)
]
trading = trading.assign(days_after_promo=pd.concat(wake_index).reindex(trading.index))

# The matched baseline: the same store's same weekday, on days neither on promotion nor
# in the week after one.
quiet = trading[(trading[s.PROMO] == 0) & (trading["days_after_promo"] < 0)]
baseline = quiet.groupby([s.STORE, s.DAY_OF_WEEK], observed=True)[s.SALES].mean()

wake = trading[trading["days_after_promo"] > 0].copy()
wake["expected"] = pd.MultiIndex.from_frame(wake[[s.STORE, s.DAY_OF_WEEK]]).map(baseline)
wake["lift"] = wake[s.SALES] / wake["expected"] - 1.0

profile = wake.groupby("days_after_promo")["lift"].agg(["mean", "count"])
on_promo = trading[trading[s.PROMO] == 1].copy()
on_promo["expected"] = pd.MultiIndex.from_frame(on_promo[[s.STORE, s.DAY_OF_WEEK]]).map(baseline)
promo_lift = float((on_promo[s.SALES] / on_promo["expected"] - 1.0).mean())

print(f"lift while the promotion runs: {promo_lift:+.1%}")
print("\nlift in the days after it ends, against the same store's same weekday")
print(profile.round(4).to_string())
print(f"\nmean over the following week: {profile['mean'].mean():+.2%}")

In [ ]:
fig, ax = plt.subplots()
ax.bar(profile.index, profile["mean"] * 100, color="#2c6fa8")
ax.axhline(0.0, color="black", linewidth=1)
ax.axhline(promo_lift * 100, color="#b03a2e", linestyle="--", linewidth=1,
           label=f"while on promotion ({promo_lift:+.0%})")
ax.set_xlabel("trading days after the promotion ended")
ax.set_ylabel("sales against a matched store-weekday baseline (%)")
ax.set_title("Q4 — does the lift persist, or was demand pulled forward?")
ax.legend()
print(plots.save(fig, "q4_promotion_wake"))
plt.show()

## Q5 — Does the classifier add anything over binning the regressor?

**Expected:** the dedicated classifier wins, because it optimises the boundary it is
scored on rather than a squared error that treats every currency unit alike.
**Counts as a no:** binning the regression prediction scores at or above the classifier,
meaning half the registry is answering a question the other half already answered.

This is the question an examiner asks about why both models exist, and `predict._label`
already relies on the answer: when the classifier abstains on a closed day, the served
label falls back to binning the number. If binning is as good everywhere, that fallback is
the whole product and the classification half is decoration.

Both models are the same estimator on the same rows, so the only thing that differs is
what they were asked to optimise.

In [ ]:
CLASSIFIER = "hist_gradient_boosting"

thresholds = fit_thresholds(train)
labelled_train = add_demand_class(train, thresholds)
labelled_test = add_demand_class(test, thresholds)
truth = labelled_test[DEMAND_CLASS_CODE]

direct = build(CLASSIFIER, task="classification", horizon=HORIZON).fit(labelled_train)
predicted_classes = direct.predict(labelled_test)

regressor = build(CLASSIFIER, task="regression", horizon=HORIZON).fit(labelled_train)
predicted_sales = regressor.predict(labelled_test)

cuts = np.array(
    [thresholds.for_store(int(store)) for store in labelled_test[s.STORE]], dtype="float64"
)
binned = (predicted_sales.to_numpy() > cuts[:, 0]).astype(int) + (
    predicted_sales.to_numpy() > cuts[:, 1]
).astype(int)
binned = pd.Series(binned, index=labelled_test.index, dtype="Int64").where(truth.notna())

answers = {
    "classifier": score_classes(truth, predicted_classes),
    "regress then bin": score_classes(truth, binned),
}
comparison = pd.DataFrame(
    {
        name: {
            "accuracy": scored.accuracy,
            "macro_f1": scored.macro_f1,
            "adjacent": scored.adjacent_accuracy,
            **{f"recall {label}": value for label, value in scored.per_class_recall.items()},
        }
        for name, scored in answers.items()
    }
)
print(comparison.round(3).to_string())

gap = answers["classifier"].macro_f1 - answers["regress then bin"].macro_f1
print(f"\nthe classifier is worth {gap:+.4f} macro-F1 over binning the regression")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
routes = (("classifier", predicted_classes), ("regress then bin", binned))
for ax, (name, prediction) in zip(axes, routes, strict=True):
    matrix = confusion(truth, prediction)
    ax.imshow(matrix.to_numpy(), cmap="Blues")
    ax.set_xticks(range(len(matrix.columns)), matrix.columns)
    ax.set_yticks(range(len(matrix.index)), list(matrix.index))
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(f"{name} — macro-F1 {answers[name].macro_f1:.3f}")
    ax.grid(False)
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            ax.text(column, row, matrix.iat[row, column], ha="center", va="center")
fig.suptitle("Q5 — two routes to the same three labels", fontweight="bold")
fig.tight_layout()
print(plots.save(fig, "q5_classifier_vs_binned"))
plt.show()

## What this notebook is not evidence of

Every chart above ran, and every number is reproducible from the committed sample with
`python scripts/make_sample.py`. None of them is a finding about retail. The generator has
a deterministic promotion calendar and a fixed weekday pattern, so a model with calendar
features is handed most of the answer — which is why the ladder is nearly flat and why the
leakage arms in [`docs/results.md`](../docs/results.md) find almost nothing.

The verdicts, including the ones that lost, are written up in
[`docs/results.md`](../docs/results.md). Answering these questions for real is one
`python -m stockout fetch` and two changed lines at the top of this notebook.